<a href="https://colab.research.google.com/github/veeranagouda-mudigoudra7/project1/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [28]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [29]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [30]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section."
        "Documents can be related to any topic - sports, manga, food, music etc."
        "Have a minimum of 4 sentences with each document."
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),
        "fill_in1": (
        "The Olympic Games are a major international sports event held every four years. Thousands of athletes from over two hundred nations participate in a variety of competitions. The event alternates between Summer and Winter Games, providing a global stage for sportsmanship. Winning a medal at the Olympics is considered one of the highest achievements in an athlete's career."
    ),
    "fill_in2": (
        "Artificial intelligence is rapidly transforming various industries around the globe. Machine learning algorithms allow computers to learn from massive amounts of data without explicit programming. Applications range from autonomous vehicles and medical diagnostics to personalized recommendations on streaming platforms. As the technology advances, ethical considerations regarding data privacy and automation become increasingly crucial."
    ),
    "fill_in3": (
        "The Amazon Rainforest is the largest tropical rainforest in the world and spans multiple South American countries. It represents over half of the planet's remaining rainforests and comprises the largest biodiversity tract. Millions of species of insects, plants, birds, and other life forms thrive within this dense ecosystem. Protecting this region is vital for global climate regulation because its vast vegetation absorbs massive quantities of carbon dioxide."
    ),
    "fill_in4": (
        "Quantum computing relies on the principles of quantum mechanics to process complex data in entirely new ways. Unlike classical computers that use binary bits representing either zero or one, quantum computers utilize qubits. Qubits can exist in a state of superposition, allowing them to calculate multiple possibilities simultaneously. This technology holds the potential to revolutionize fields like cryptography, molecular modeling, and complex systems optimization."
    )

}

In [31]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))

[Source: sample_protocol]
Students must fill in documents in this section.Documents can be related to any topic - sports, manga, food, music etc.Have a minimum of 4 sentences with each document.Duplicate this for a minimum of 4 documents to test your agent.

[Source: fill_in4]
Quantum computing relies on the principles of quantum mechanics to process complex data in entirely new ways. Unlike classical computers that use binary bits representing either zero or one, quantum computers utilize qubits. Qubits can exist in a state of superposition, allowing them to calculate multiple possibilities simultaneously. This technology holds the potential to revolutionize fields like cryptography, molecular modeling, and complex systems optimization.


In [32]:
# @title Ask gemini to search for something
import google.generativeai as genai
import os  # Added missing import
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-3.8-flash",
    tools=[search_documents],  # Fixed function name to match your defined tool
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.8-flash


In [33]:
import time
from google import genai
from google.genai import types

client = genai.Client()

# 1. Pass the Python function directly into tools.
# The SDK automatically infers the name, description, and JSON schema from your docstring.
config = types.GenerateContentConfig(
    tools=[search_documents],
    # 2. Let the SDK handle the multi-turn tool call loop natively
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False),
    # Optional: Add your Michelin chef or student counselor instructions here
    system_instruction="Only answer based on what you find using the search_documents tool.",
)

def run_agent_simplified(question: str) -> str:
    print(f"Question: {question}\n" + "-" * 50)

    # The client handles all tool calls behind the scenes and yields the final response
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=config
    )

    print(f"\nFinal answer:\n{response.text}")
    return response.text

_ = run_agent_simplified("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    # This question can be resolved by searching the recipe database
    "What is the exact cooking temperature and time required for the beef tenderloin?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)